In [1]:
import pandas as pd 
import numpy as np 
import yfinance as yf 
import vectorbt as vbt 
import talib



### MACD for ticker - "AAPL"

In [2]:
df = yf.download(tickers = "AAPL", start= "2015-05-05", end= "2025-05-05")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

close = df["Close"]

macd, signal, hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)

entries = macd >= signal
exits = macd <= signal

backtest = vbt.Portfolio.from_signals(
       close,
       entries = entries,
       exits = exits,
       init_cash = 100000,
       size = 10000.0,
       size_type = "value",
       freq = "1D", 
       fees = 0.001)

backtest.plot().show()
print(backtest.stats())

df_trades = backtest.trades.records_readable
print(df_trades)

[*********************100%***********************]  1 of 1 completed


Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                  100000.0
End Value                               120402.446905
Total Return [%]                            20.402447
Benchmark Return [%]                       630.922635
Max Gross Exposure [%]                      11.445011
Total Fees Paid                           1731.975621
Max Drawdown [%]                             2.920416
Max Drawdown Duration               215 days 00:00:00
Total Trades                                       86
Total Closed Trades                                85
Total Open Trades                                   1
Open Trade PnL                             148.801191
Win Rate [%]                                51.764706
Best Trade [%]                               28.52362
Worst Trade [%]                            -12.056632
Avg Winning Trade [%]       

### MACD for 10 companies in the S and P 500 (different sectors)

In [8]:
#For multiple companies


df1 = yf.download(tickers = ["AAPL", "GOOG", "AMZN", "MSFT", "TXN", "MU", "TXN", "RTX", "NVDA", "TSLA"], start= "2015-05-05", end= "2025-05-05", threads = True)


close_df = df1["Close"].dropna(axis=1, how="all")
high_df = df1["High"].dropna(axis=1, how="all")
low_df = df1["Low"].dropna(axis=1, how="all")


valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns)
)
close_df, high_df, low_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
)

entries = macd >= signal
exits = macd <= signal

backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001
)


print(backtest1.stats())
backtest1.plot(subplots = ["value","cash"]).show()

[*********************100%***********************]  9 of 9 completed


Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                  100000.0
End Value                               247642.526608
Total Return [%]                           147.642527
Benchmark Return [%]                      2974.153856
Max Gross Exposure [%]                      65.511147
Total Fees Paid                          20770.624392
Max Drawdown [%]                             16.08959
Max Drawdown Duration               383 days 00:00:00
Total Trades                                      891
Total Closed Trades                               882
Total Open Trades                                   9
Open Trade PnL                           10332.995513
Win Rate [%]                                41.723356
Best Trade [%]                             122.872188
Worst Trade [%]                            -22.532062
Avg Winning Trade [%]       

### MACD for 500 companies in the S and P 500

In [10]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2025-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")


valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns)
)
close_df, high_df, low_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
)


macd, signal, hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)


entries = (macd >= signal)
exits = (macd < signal)



backtest5 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
    tp_stop = 0.10
       
)


print(backtest5.stats())
backtest5.plot().show()


[****                   8%                       ]  39 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[**********            20%                       ]  98 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[**********************45%                       ]  225 of 500 completed$Q: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[**********************93%********************   ]  466 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[*********************100%***********************]  500 of 500 co

Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                  100000.0
End Value                               249051.149274
Total Return [%]                           149.051149
Benchmark Return [%]                       337.427307
Max Gross Exposure [%]                          100.0
Total Fees Paid                          37649.276147
Max Drawdown [%]                            18.222503
Max Drawdown Duration               207 days 00:00:00
Total Trades                                    46716
Total Closed Trades                             46268
Total Open Trades                                 448
Open Trade PnL                            7509.215297
Win Rate [%]                                52.997752
Best Trade [%]                              74.318649
Worst Trade [%]                            -57.888236
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### MACD and RSI hybrid for 500 companies in the S and P 500

In [13]:
macd, signal, hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
rsi = talib.RSI(close, timeperiod=14)

entries = (macd >= signal) & (rsi >= 30)
exits = (macd < signal) | (rsi > 70)



backtest6 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
    tp_stop = 0.10
       
)


print(backtest6.stats())
backtest6.plot().show()

Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                100,000.00
End Value                                  234,884.59
Total Return [%]                               134.88
Benchmark Return [%]                           337.43
Max Gross Exposure [%]                         100.00
Total Fees Paid                             32,515.94
Max Drawdown [%]                                19.44
Max Drawdown Duration               420 days 00:00:00
Total Trades                                    41719
Total Closed Trades                             41271
Total Open Trades                                 448
Open Trade PnL                               7,081.87
Win Rate [%]                                    52.19
Best Trade [%]                                  74.32
Worst Trade [%]                                -57.89
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



In [15]:
pd.options.display.float_format = "{:,.2f}".format

f = backtest6.trades.records_readable


,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
0,0,A,272.69,2015-07-15,36.64,9.99,2015-07-27,36.16,9.86,-149.30,-0.01,Long,Closed,0
1,1,A,275.93,2015-08-18,35.62,9.83,2015-08-20,34.24,9.45,-399.65,-0.04,Long,Closed,1
2,2,A,284.56,2015-08-28,33.33,9.48,2015-09-29,30.89,8.79,-712.41,-0.08,Long,Closed,2
3,3,A,274.07,2015-10-12,32.95,9.03,2015-11-11,34.48,9.45,400.57,0.04,Long,Closed,3
4,4,A,263.28,2016-01-22,34.87,9.18,2016-04-08,36.41,9.58,385.96,0.04,Long,Closed,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41714,41714,YUM,0.00,2024-11-26,133.39,0.00,2024-12-30,129.10,0.00,-0.00,-0.03,Long,Closed,41714
41715,41715,YUM,0.00,2025-01-31,126.18,0.00,2025-02-06,139.24,0.00,0.00,0.10,Long,Closed,41715
41716,41716,YUM,2.07,2025-02-07,138.80,0.29,2025-03-03,152.45,0.32,27.66,0.10,Long,Closed,41716
41717,41717,YUM,0.00,2025-03-28,151.31,0.00,2025-04-03,156.84,0.00,0.00,0.03,Long,Closed,41717


In [16]:
f.describe()

,Exit Trade Id,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Position Id
count,"41,719.00","41,719.00",41719,"41,719.00","41,719.00",41719,"41,719.00","41,719.00","41,719.00","41,719.00","41,719.00"
mean,"20,859.00",6.50,2020-10-07 10:31:22,116.99,0.39,2020-10-24 11:34:16,117.75,0.39,3.23,0.01,"20,859.00"
min,0.00,0.00,2015-07-15 00:00:00,0.48,0.00,2015-07-16 00:00:00,0.47,0.00,"-3,232.59",-0.58,0.00
25%,"10,429.50",0.00,2018-05-02 00:00:00,35.82,0.00,2018-05-25 00:00:00,36.10,0.00,-0.00,-0.03,"10,429.50"
50%,"20,859.00",0.00,2020-11-11 00:00:00,69.24,0.00,2020-11-23 00:00:00,69.76,0.00,0.00,0.00,"20,859.00"
75%,"31,288.50",0.60,2023-05-05 00:00:00,131.39,0.05,2023-05-16 00:00:00,132.28,0.04,0.00,0.05,"31,288.50"
max,"41,718.00",871.21,2025-05-02 00:00:00,"9,768.29",24.09,2025-05-02 00:00:00,"9,421.74",26.62,"4,128.89",0.74,"41,718.00"
std,"12,043.38",30.37,NaN,254.46,1.77,NaN,253.81,1.78,114.17,0.07,"12,043.38"
